# 0305 · Guías con destino Bogotá, incluidas las abiertas (ajuste C1)

**Por qué.** 0300 (celda 12) filtraba `ID_ESTADOENVIO >= 3`, que deja solo guías cerradas (entregadas, devueltas o siniestradas) y saca las que seguían abiertas al extraer. Las más lentas del final del periodo quedaban fuera.

**Qué hace.** Escribe un archivo **nuevo**, `LOG_GUIAS/RAW_V4/LOG_GUIAS_BOGOTA_V4.parquet`, con las mismas columnas y la misma minimización de datos que 0300 v8 (nombres, teléfonos y direcciones en nulo; receptor solo como indicador; `ID_CLIENTE` seudonimizado). Solo cambian dos cosas:

1. El filtro de estado pasa a una **lista explícita** (`ESTADOS_ENVIO_INCLUIDOS`), que se fija con la tabla `P10_1c_guias_mes_estado.csv` de 0345. Si en los datos aparece un estado que no está ni en la lista ni en `ESTADOS_EXCLUIDOS`, el cuaderno se detiene.
2. Una sola fila por guía: si una guía aparece en dos lotes, se queda la extracción más reciente.

El archivo de 0300 (`LOG_GUIAS/RAW/LOG_GUIAS_BOGOTA.parquet`) no se toca: la base v3 sigue siendo reproducible.

**Sal del cliente.** Se lee de `SEGURIDAD/sal_cliente.txt`. Si se guarda allí la misma sal que se usó en 0300, los seudónimos del cliente coinciden con los de la v3 (la celda de control lo dice). Si no se conoce, se crea una nueva; los seudónimos serán distintos de la v3, pero coherentes dentro de la v4, que es lo que necesita el modelo.

In [ ]:
%pip install "duckdb<2"

In [ ]:
import os, secrets, duckdb, pandas as pd
PATH_DATABRICKS = "/Volumes/proyectogrado/source/source/"
RUTA_BATCH_LOG_GUIAS = f"{PATH_DATABRICKS}LOG_GUIAS/LOG_GUIAS*.parquet"                    # 0100 (crudos)
RUTA_GUIAS_V3        = f"{PATH_DATABRICKS}LOG_GUIAS/RAW/LOG_GUIAS_BOGOTA.parquet"           # 0300 (no se toca)
RUTA_GUIAS_V4        = f"{PATH_DATABRICKS}LOG_GUIAS/RAW_V4/LOG_GUIAS_BOGOTA_V4.parquet"     # salida de este cuaderno
RUTA_EST_ENVIO       = f"{PATH_DATABRICKS}PARAMETRICAS/PAR.ESTADOSENVIO.parquet"
RUTA_SAL_CLIENTE     = f"{PATH_DATABRICKS}SEGURIDAD/sal_cliente.txt"

# Fijar con P10_1c_guias_mes_estado.csv (0345). Propuesta del director: 1 a 5
# (1 recibido, 2 en procesamiento, 3 entregado, 4 entregado a remitente, 5 siniestrado).
# 9 (en alistamiento del cliente) se mantiene porque 0300 (>= 3) ya lo incluía: así el único cambio frente a la v3
# es sumar las abiertas 1 y 2. Las que no tengan salida a reparto ni admisión válida salen en 0330 (sin T0), con conteo.
ESTADOS_ENVIO_INCLUIDOS = [1, 2, 3, 4, 5, 9]
ESTADOS_EXCLUIDOS       = {}          # {id: "razón"} para estados que aparezcan y no deban entrar (p. ej., anulada)
# Las guías con ID_ESTADOENVIO nulo no tienen estado: se excluyen siempre y se cuentan en el control previo.

In [ ]:
# GUARDA DE LA RESERVA (C2): septiembre y octubre no se leen hasta registrar P4b_decision_v2.json.
# Si en las carpetas de desarrollo hay archivos de 202609 o posteriores, el cuaderno se detiene sin abrirlos.
import glob as _glob, re as _re, os as _os
_reserva = []
for _carpeta in ["LOG_GUIAS", "LOG_SEGUIMIENTO_ENVIO", "LOG_DOCUMENTOS"]:
    for _f in _glob.glob(f"{PATH_DATABRICKS}{_carpeta}/*.parquet"):
        _m = _re.search(r"(20\d{4})\D*\.parquet$", _os.path.basename(_f))
        if _m and int(_m.group(1)) >= 202609:
            _reserva.append(f"{_carpeta}/{_os.path.basename(_f)}")
assert not _reserva, ("Hay archivos de la reserva en las carpetas de desarrollo: " + ", ".join(sorted(_reserva)) +
                      ". Moverlos sin abrirlos a RESERVA_POSTERIOR/<misma carpeta>/ (celda del LEEME) y volver a correr.")
print("Guarda de la reserva: no hay archivos de septiembre en adelante en las carpetas de desarrollo.")

In [ ]:
# ESCRITURA EN EL VOLUME · en Databricks serverless el FUSE de /Volumes no acepta las escrituras
# asíncronas de DuckDB ("Async write failed ... errno 5"): el COPY escribe en /tmp y luego copia.
import os, re, shutil, tempfile, uuid
def copy_a_volume(sql_copy, con=None):
    """Ejecuta un COPY (...) TO '<ruta>' de DuckDB escribiendo en /tmp y copiando el resultado a <ruta>."""
    destino_m = list(re.finditer(r"\bTO\s+'([^']+)'", sql_copy, flags=re.IGNORECASE))[-1]
    destino = destino_m.group(1)
    local = os.path.join(tempfile.gettempdir(), f"duckdb_{uuid.uuid4().hex}_{os.path.basename(destino)}")
    sql_local = sql_copy[:destino_m.start(1)] + local + sql_copy[destino_m.end(1):]
    try:
        (con or duckdb).execute(sql_local)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        try:
            shutil.copyfile(local, destino)
        except OSError:
            dbutils.fs.cp(f"file:{local}", destino)
    finally:
        if os.path.exists(local):
            os.remove(local)
    print(f"Escrito en {destino}")
    return destino

In [ ]:
# Control previo: todo estado presente debe estar incluido o excluido con razón
presentes = duckdb.sql(f"""SELECT G.ID_ESTADOENVIO, any_value(E.NOMBRE_ESTADOENVIO) AS nombre, COUNT(*) AS guias
                          FROM read_parquet('{RUTA_BATCH_LOG_GUIAS}') G
                          LEFT JOIN read_parquet('{RUTA_EST_ENVIO}') E USING (ID_ESTADOENVIO)
                          WHERE G.ID_CIUDAD_DESTINO = 10 AND length(G.NUMERO_GUIA::VARCHAR) >= 8
                          GROUP BY 1 ORDER BY 1""").df()
display(presentes)
nulos = int(presentes.loc[presentes.ID_ESTADOENVIO.isna(), "guias"].sum())
print(f"Guías con estado nulo (se excluyen): {nulos:,}")
sin_decidir = set(presentes.ID_ESTADOENVIO.dropna().astype(int)) - set(ESTADOS_ENVIO_INCLUIDOS) - set(ESTADOS_EXCLUIDOS)
assert not sin_decidir, f"Estados presentes sin decidir: {sorted(sin_decidir)}. Agregarlos a ESTADOS_ENVIO_INCLUIDOS o a ESTADOS_EXCLUIDOS"

In [ ]:
def leer_o_crear_sal(ruta):
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        with open(ruta, "w") as f: f.write(secrets.token_hex(16))
        print(f"Sal nueva creada en {ruta}: los seudónimos de cliente no coincidirán con los de la v3.")
    with open(ruta) as f: sal = f.read().strip()
    assert len(sal) >= 16 and "'" not in sal, "La sal debe tener al menos 16 caracteres y no llevar comillas"
    return sal

_sal_cliente = leer_o_crear_sal(RUTA_SAL_CLIENTE)
lista = ", ".join(str(int(x)) for x in ESTADOS_ENVIO_INCLUIDOS)
con = duckdb.connect()
copy_a_volume(f"""
    COPY (
        SELECT 
            NUMERO_GUIA,ID_TIPOGUIA,md5(CAST(ID_CLIENTE AS VARCHAR) || '{_sal_cliente}') AS ID_CLIENTE,ID_CANAL,ID_DIVISION_PROV,ID_1,ID_2,ID_PAIS_ORIGEN,ID_CIUDAD_ORIGEN,
            ID_PAIS_DESTINO,ID_CIUDAD_DESTINO,ID_TIPO_ZONA_URBA,ID_PAIS_ZONA_URB_ENTR,ID_CIUDAD_ZONA_URB_ENTR,ID_ZONA_URBA,ID_ESTADOENVIO,
            ID_ESTADOGUIA,ID_CARACT_ENVIO,ID_TIEMPO,ID_MEDTRANS,ID_PRODUCTO,ID_SUBPRODUCTO,ID_BODEPROV,ID_PAIS_UBICACION,ID_CIUDAD_UBICACION,
            ID_DIVISION_UBICACION,FECHA_ENVIO,NULL AS ID_REMITE, NULL AS ID_DESTINA,null as  NOMBRE_REMITE_GUIA,null as  APELLIDO_REMITE_GUIA,
            null as  NOMBRE_DESTINA_GUIA,null as  APELLIDO_DESTINA_GUIA,null as  TELEFONO_REMITE_GUIA,null as  TELEFONO_DESTINA_GUIA,
            null as  DIRECCION_REMITE_GUIA,null as  DIRECCION_DESTINA_GUIA,NULL AS IDEN_RECEP_SERVI, NULL AS IDEN_ENTR_SERVI, NULL AS CONTENIDO,LARGO_ENVIO,
            ALTO_ENVIO,ANCHO_ENVIO,LARGO_ENV_OPERA,ALTO_ENV_OPERA,ANCHO_ENV_OPERA,PESO_GUIA,PESO_FISICO_GUIA,PESO_VOLU_GUIA,
            PESO_FISICO_OPERA,PESO_VOLU_OPERA,PESO_FACTURADO,FECHA_ENTREGA,NUMERO_PIEZAS,CASE WHEN NULLIF(TRIM(CAST(IDEN_PERS_RECIBE AS VARCHAR)), '') IS NOT NULL THEN 'S' END AS IDEN_PERS_RECIBE, CASE WHEN NULLIF(TRIM(CAST(NOMB_PERS_RECIBE AS VARCHAR)), '') IS NOT NULL THEN 'S' END AS NOMB_PERS_RECIBE,
            ID_REMPLAZA,NULL AS IDEN_DEPOSITANTE, NULL AS NOMB_DEPOSITANTE,
            FECHA_ULTIMA_MOD,NULL AS DIRECCION_ESTANDAR, NULL AS OBSERVACIONES,GUIA_EXTERNA,ID_DOCU_ULT_MOVI,NU_DOCU_ULT_MOVI,CONS_DOCU_ULT_MOVI,ID_FACTURADO,VALOR_DECLARADO_GUIA,VALOR_SEG_MAN_GUIAS,VALOR_FLETES_GUIAS,VALOR_DESCUENTO_GUIAS,VALOR_PEDIDO_GUIAS,VALOR_SEG_MAN_GUIAS_FACT,VALOR_FLETES_GUIAS_FACT,ID_REQUIERE_EMBALAJE,null as  LOGIN_EMPLEADO,FECHA_CREACION,FECHA_MODIFICACION,FECHA_MODIFICACION_DG,FECHA_EXTRACCION_DW
        FROM read_parquet('{RUTA_BATCH_LOG_GUIAS}')
        WHERE ID_CIUDAD_DESTINO = 10
          AND LEN(NUMERO_GUIA::VARCHAR) >= 8
          AND ID_ESTADOENVIO IN ({lista})
        QUALIFY ROW_NUMBER() OVER (PARTITION BY NUMERO_GUIA ORDER BY FECHA_EXTRACCION_DW DESC, FECHA_ULTIMA_MOD DESC) = 1
    ) TO '{RUTA_GUIAS_V4}' (FORMAT PARQUET, COMPRESSION ZSTD)""", con)
con.close()
del _sal_cliente

In [ ]:
# Controles: una fila por guía, conteo por estado y comparación con el archivo de 0300 (solo conteos)
c = duckdb.sql(f"""
    WITH V4 AS (SELECT * FROM read_parquet('{RUTA_GUIAS_V4}')), V3 AS (SELECT NUMERO_GUIA, ID_CLIENTE FROM read_parquet('{RUTA_GUIAS_V3}'))
    SELECT (SELECT COUNT(*) FROM V4) AS filas_v4, (SELECT COUNT(DISTINCT NUMERO_GUIA) FROM V4) AS guias_v4,
           (SELECT COUNT(*) FROM V3) AS filas_v3,
           (SELECT COUNT(*) FROM V4 WHERE ID_ESTADOENVIO < 3) AS abiertas_nuevas,
           (SELECT COUNT(*) FROM V3 WHERE NUMERO_GUIA NOT IN (SELECT NUMERO_GUIA FROM V4)) AS v3_que_no_estan_en_v4,
           (SELECT AVG((V4.ID_CLIENTE = V3.ID_CLIENTE)::INT) FROM V4 JOIN V3 USING (NUMERO_GUIA)) AS pct_cliente_mismo_seudonimo""").df()
display(c)
assert c.loc[0, "filas_v4"] == c.loc[0, "guias_v4"], "Hay guías repetidas en la salida"
if c.loc[0, "pct_cliente_mismo_seudonimo"] < 0.999:
    print("AVISO: la sal del cliente no es la de 0300. La v4 es coherente por dentro, pero el seudónimo del cliente cambia frente a la v3.")
display(duckdb.sql(f"""SELECT strftime(FECHA_ENVIO, '%Y-%m') AS mes, ID_ESTADOENVIO, COUNT(*) AS guias
                       FROM read_parquet('{RUTA_GUIAS_V4}') GROUP BY ALL ORDER BY ALL""").df()
       .pivot_table(index="mes", columns="ID_ESTADOENVIO", values="guias", fill_value=0))

## Qué sigue

Con este archivo se corre `0330_BaseAnaliticaV4` (que lo lee en `RUTA_GUIAS`). No hace falta volver a correr el resto de 0300: 0330 lee los movimientos y documentos crudos.